# Exploratory Analysis — Map of Italian Science

This notebook contains the exploratory visual analysis for the project on the international citation relationships of six Italian institutions.

Goals:
- explore inbound/outbound citation distributions
- compare institutions
- identify dominant countries and organizations
- test candidate visualizations for the final paper

In [28]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

BASE_PATH = Path("data/citation_counts")

INSTITUTIONS = {
    "UNIBO": BASE_PATH / "UNIBO",
    "UNIMI": BASE_PATH / "UNIMI",
    "UNIPD": BASE_PATH / "UNIPD",
    "UNITO": BASE_PATH / "UNITO",
    "UPO": BASE_PATH / "UPO",
    "SNS": BASE_PATH / "SNS"
}

In [35]:
def load_country_data(institution, direction):
    path = (
        INSTITUTIONS[institution]
        / f"citation_counts_countries_{direction}.csv"
    )

    return pd.read_csv(path)

inbound_df = load_country_data("UNIBO", "inbound")

inbound_df.head()

,country_name,country_code,count
0,United States,US,6800261
1,France,FR,3956305
2,Italy,IT,3643907
3,United Kingdom,GB,1986678
4,China,CN,1986405


## Initial Inspection of Country-Level Data: inbound

In [36]:
# Which countries dominate?
inbound_df.sort_values("count", ascending=False).head(10)

,country_name,country_code,count
0,United States,US,6800261
1,France,FR,3956305
2,Italy,IT,3643907
3,United Kingdom,GB,1986678
4,China,CN,1986405
5,Germany,DE,1801165
6,Spain,ES,1262588
7,Japan,JP,889572
8,Canada,CA,676387
9,Australia,AU,661316


In [37]:
top10 = inbound_df.sort_values(
    "count",
    ascending=False
).head(10)

fig = px.bar(
    top10,
    x="count",
    y="country_name",
    orientation="h",
    title="Top 10 Inbound Countries — UNIBO"
)

fig.show()

### Observation

UNIBO inbound citations appear strongly concentrated in:
- United States
- France
- Italy

The distribution is highly skewed: Uniated States alone has 6.8 million, enormously larger compared to others. This suggest a very strong dependance on US-cenered citation ecosystem. 

Also Italy appears in the inbound dataset, so we should decide if we want to keep Italy in the dataset or not. We want to focus on international relations, so we should not include Italy, otherwise it would obscure international patterns. 

In [38]:
inbound_df = inbound_df[inbound_df["country_code"] != "IT"]

top15 = inbound_df.sort_values(
    "count",
    ascending=False
).head(15)

fig = px.bar(
    top15,
    x="count",
    y="country_name",
    orientation="h",
    title="Top 10 Inbound Countries — UNIBO"
)

fig.show()

Even thouhg we dropped Italy, it seems a classic **long-tail distribution**, in which we have few dominant countries (United States and France) and many countries with tiny counts.

### Important implication for visualization 
A choropleth map may:

- visually flatten differences,
- because the US dominates so strongly.

You may later need:
- logarithmic scaling,
- percentile scaling,
- or capped color ranges.

This is a VERY important discovery from exploration.


## Compare inbound/outbound for UNIBO

Questions:
* is outbound also US-dominated?
* is outbound more geographically diverse?

In [39]:
outbound_df = load_country_data("UNIBO", "outbound")

outbound_df.head(10)

,country_name,country_code,count
0,United States,US,8894677
1,France,FR,4043901
2,Italy,IT,3669866
3,United Kingdom,GB,2461117
4,Germany,DE,1828283
5,Spain,ES,1051307
6,Japan,JP,842595
7,China,CN,792802
8,Canada,CA,761331
9,The Netherlands,NL,705264


In [40]:
outbound_df = outbound_df[outbound_df["country_code"] != "IT"]

top15 = outbound_df.sort_values(
    "count",
    ascending=False
).head(15)

fig = px.bar(
    top15,
    x="count",
    y="country_name",
    orientation="h",
    title="Top 10 Outbound Countries — UNIBO"
)

fig.show()

### Findings

**United States** dominate both directions:
* Inbound 6.8M
* Outbound 8.9M

Italian insitutions cite US research even more. 
This suggest strong epistemic dependence on US-centered science. 

**France** is very symmetric in inbound and outbound.
- inbound 3.95M
- outbound 4.04M

This suggests a relatively reciprocal citation relationship. 

**China** changes dramatically:
- inbound 1.98M
- outbound 0.79M

Chinese research cites Italian science heabily but Italian institutions cite Chinese research much less. 
Potential interpretations:
- language/publication ecosystem differences,
- citation prestige asymmetries,
- disciplinary differences,
- Western-centric citation behavior.

**The Netherlands** appears only in outbound top 10. 


In [66]:
inbound_renamed = inbound_df.rename(
    columns={"count": "inbound_count"}
)

outbound_renamed = outbound_df.rename(
    columns={"count": "outbound_count"}
)

# Merge the two dataframes
merged_df = pd.merge(
    inbound_renamed,
    outbound_renamed,
    on=["country_name", "country_code"],
    how="outer"
)

# Replace missing values 
merged_df = merged_df.fillna(0)

# Create asymmetry metrics
merged_df["difference"] = (
    merged_df["outbound_count"]
    - merged_df["inbound_count"]
)

merged_df["total"] = (
    merged_df["inbound_count"]
    + merged_df["outbound_count"]
)

merged_df.head(10)


,country_name,country_code,inbound_count,outbound_count,difference,total
0,Afghanistan,AF,432.0,95.0,-337.0,527.0
1,Albania,AL,1403.0,754.0,-649.0,2157.0
2,Algeria,DZ,9429.0,3454.0,-5975.0,12883.0
3,American Samoa,AS,1104.0,365.0,-739.0,1469.0
4,Andorra,AD,19.0,5.0,-14.0,24.0
5,Angola,AO,1220.0,805.0,-415.0,2025.0
6,Antigua and Barbuda,AG,1952.0,568.0,-1384.0,2520.0
7,Argentina,AR,111735.0,96794.0,-14941.0,208529.0
8,Armenia,AM,24520.0,26290.0,1770.0,50810.0
9,Aruba,AW,7.0,5.0,-2.0,12.0


In [68]:
top15_countries = merged_df.sort_values(
    "total",
    ascending=False
).head(15)

top15_countries.head(10)

,country_name,country_code,inbound_count,outbound_count,difference,total
239,United States,US,6800261.0,8894677.0,2094416.0,15694938.0
76,France,FR,3956305.0,4043901.0,87596.0,8000206.0
238,United Kingdom,GB,1986678.0,2461117.0,474439.0,4447795.0
81,Germany,DE,1801165.0,1828283.0,27118.0,3629448.0
42,China,CN,1986405.0,792802.0,-1193603.0,2779207.0
204,Spain,ES,1262588.0,1051307.0,-211281.0,2313895.0
108,Japan,JP,889572.0,842595.0,-46977.0,1732167.0
36,Canada,CA,676387.0,761331.0,84944.0,1437718.0
10,Australia,AU,661316.0,670465.0,9149.0,1331781.0
223,The Netherlands,NL,587860.0,705264.0,117404.0,1293124.0


In [72]:
# Inbound rows
inbound_long = top15_countries[[
    "country_name",
    "inbound_count"
]].copy()

inbound_long["direction"] = "inbound"

inbound_long["value"] = (
    -inbound_long["inbound_count"]
)

inbound_long.head(10)

,country_name,inbound_count,direction,value
239,United States,6800261.0,inbound,-6800261.0
76,France,3956305.0,inbound,-3956305.0
238,United Kingdom,1986678.0,inbound,-1986678.0
81,Germany,1801165.0,inbound,-1801165.0
42,China,1986405.0,inbound,-1986405.0
204,Spain,1262588.0,inbound,-1262588.0
108,Japan,889572.0,inbound,-889572.0
36,Canada,676387.0,inbound,-676387.0
10,Australia,661316.0,inbound,-661316.0
223,The Netherlands,587860.0,inbound,-587860.0


In [73]:
# Outbound rows
outbound_long = top15_countries[[
    "country_name",
    "outbound_count"
]].copy()

outbound_long["direction"] = "outbound"

outbound_long["value"] = (
    outbound_long["outbound_count"]
)

outbound_long.head(10)

,country_name,outbound_count,direction,value
239,United States,8894677.0,outbound,8894677.0
76,France,4043901.0,outbound,4043901.0
238,United Kingdom,2461117.0,outbound,2461117.0
81,Germany,1828283.0,outbound,1828283.0
42,China,792802.0,outbound,792802.0
204,Spain,1051307.0,outbound,1051307.0
108,Japan,842595.0,outbound,842595.0
36,Canada,761331.0,outbound,761331.0
10,Australia,670465.0,outbound,670465.0
223,The Netherlands,705264.0,outbound,705264.0


In [77]:
diverging_df = pd.concat([
    inbound_long.rename(
        columns={"inbound_count": "count"}
    ),

    outbound_long.rename(
        columns={"outbound_count": "count"}
    )
])

# Diverging Bar Chart
fig = px.bar(
    diverging_df.sort_values("value"),

    x="value",
    y="country_name",

    color="direction",

    orientation="h",

    custom_data=["count", "direction"],

    color_discrete_map={
        "inbound": "#B7990D",
        "outbound": "#320E3B"
    },

    title="Inbound vs Outbound Citation Relationships"
)

fig.update_traces(
    hovertemplate=
    "<b>%{y}</b><br>" +
    "Citations: %{customdata[0]:,.0f}<br>" +
    "Direction: %{customdata[1]}<extra></extra>"
)

max_value = diverging_df["value"].abs().max()

fig.update_layout(
    title_x=0.5,
    template="plotly_white",
    height=600,
    bargap=0.1,
    legend_title_text=""
)

fig.update_xaxes(
    range=[-max_value * 1.05, max_value * 1.05]
)

fig.add_vline(
    x=0,
    line_width=1.5,
    line_color="gray"
)

fig.show()